# CDD-11-30: A3-M multiscale degradation reasoning

A3-M is the single planned architecture rescue after A3-L failed to recognize snow and did not beat the matched controls. It keeps the A3-L schedule and loss weights, but forms the degradation descriptor from all encoder scales and balances BCE from train-split label frequencies. It evaluates separate best-PSNR and best-macro-F1 checkpoints. The evaluator also performs a full-validation intervention-utility audit: image, 32x32-block, and 8x8-block oracles plus a fixed residual-strength control.

Import this notebook from GitHub and attach only the existing `cdd-11-30` and `nafnetmodel` Kaggle inputs. The test split remains untouched.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)
print("CWD:", Path.cwd())

In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
EXPERIMENTS_ROOT = Path("/kaggle/working/experiments_a3m")
CONFIG = Path("configs/calibration_a3_multiscale.json")
RUN_NAME = "a3m_multiscale_degradation_sidd32_seed42_20ep"
assert CDD11_ROOT.is_dir(), f"Missing CDD-11 input: {CDD11_ROOT}"
assert PRETRAINED_ROOT.is_dir(), f"Missing pretrained input: {PRETRAINED_ROOT}"
assert CONFIG.is_file(), f"Missing config: {CONFIG}"
gpu_names = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name", "--format=csv,noheader"
], text=True).strip().splitlines()
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), (
    f"Select the Kaggle 2xT4 accelerator; found: {gpu_names}"
)
print("GPUs:", gpu_names)
print("Pretrained files:", sorted(path.name for path in PRETRAINED_ROOT.glob("*.pth")))

In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT),
    "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/cot_nafnet_audit/audit_a3m.json",
], check=True)

In [ ]:
# Keep False for the first Run All; enable only after the audit and dry run pass.
RUN_A3M = False

In [ ]:
runner_command = [
    "python", "-m", "hybrid_cot_nafnet.run_ablation",
    "--config", str(CONFIG),
    "--data-root", str(CDD11_ROOT),
    "--experiments-root", str(EXPERIMENTS_ROOT),
    "--nproc-per-node", "2",
    "--runs", RUN_NAME,
]
if RUN_A3M:
    subprocess.run(runner_command, check=True)
else:
    subprocess.run([*runner_command, "--dry-run"], check=True)
    print("Dry run complete. Set RUN_A3M = True and rerun from this cell.")

In [ ]:
from IPython.display import display
import pandas as pd

run_dir = EXPERIMENTS_ROOT / RUN_NAME
for label, path in {
    "training": run_dir / "run_summary.json",
    "best restoration": run_dir / "evaluation" / "summary.json",
    "best reasoning": run_dir / "reasoning_evaluation" / "summary.json",
}.items():
    if path.is_file():
        print(f"\n{label}:")
        display(json.loads(path.read_text()))
if (run_dir / "train_log.csv").is_file():
    display(pd.read_csv(run_dir / "train_log.csv"))

In [ ]:
import zipfile
from IPython.display import FileLink, FileLinks

summary_csv = EXPERIMENTS_ROOT / "ablation_summary.csv"
required = [
    run_dir / "run_summary.json",
    run_dir / "evaluation" / "summary.json",
    run_dir / "reasoning_evaluation" / "summary.json",
]
complete = all(path.is_file() for path in required)
if complete:
    training = json.loads(required[0].read_text())
    complete = training.get("status") == "completed" and training.get("completed_epochs") == 20
if complete and summary_csv.is_file():
    display(pd.read_csv(summary_csv))
    lightweight = Path("/kaggle/working/a3m_lightweight_results.zip")
    with zipfile.ZipFile(lightweight, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
        for path in sorted(EXPERIMENTS_ROOT.rglob("*")):
            if path.is_file() and path.suffix.lower() not in {".pt", ".png", ".jpg", ".jpeg"}:
                output_zip.write(path, path.relative_to(EXPERIMENTS_ROOT))
    comparisons = Path("/kaggle/working/a3m_comparisons.zip")
    with zipfile.ZipFile(comparisons, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
        for evaluation_name in ("evaluation", "reasoning_evaluation"):
            for path in sorted((run_dir / evaluation_name / "comparisons").glob("*.png")):
                output_zip.write(path, Path(evaluation_name) / path.name)
    display(FileLink(str(lightweight)))
    display(FileLink(str(comparisons)))
else:
    print("A complete 20-epoch run and both evaluations are required before export.")
if EXPERIMENTS_ROOT.is_dir():
    display(FileLinks(str(EXPERIMENTS_ROOT)))